# Detectie & Tuning – Dataset B

Kwantitatieve evaluatie van verschillende AI-architecturen (CNN vs. Transformer) en optimalisatie van hyperparameters (Confidence Threshold, Lost Track Buffer) voor loperdetectie.

> **Vereisten:**
> - Video `Marathon.mp4` (verkrijgbaar via mail)
> - Roboflow API-key (verkrijgbaar via mail)
> - Ground Truth telling: 280 lopers
>
> **Output:** Benchmark-statistieken in de terminal en een visueel validatie-frame.

In [ ]:
import cv2
import time
import supervision as sv
from inference import get_model
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os

print("Bibliotheken succesvol geïnstalleerd en geïmporteerd.")

# Globale Constanten en Configuratie
VIDEO_PATH = '../../data/video/Marathon.mp4'

# VUL HIER JE ROBOFLOW API KEY IN
ROBOFLOW_API_KEY = "API_KEY"
GROUND_TRUTH_COUNT = 280

print(f"Paden geconfigureerd. Ground Truth ingesteld op: {GROUND_TRUTH_COUNT} lopers.")

## Architectuur Benchmark & Hyperparameter Tuning

In [ ]:
print("Start met de kwantitatieve model-evaluatie...")

# Te evalueren modellen (Transfer Learning via Roboflow)
MODELS_TO_TEST = {
    # Architectuur Test (Lichtgewicht Modellen)
    "RF-DETR (Nano)": "tracking-runners/5",
    "YOLOv11 (Fast)": "tracking-runners/8",
    "YOLOv12 (Fast)": "tracking-runners/9",
    "YOLO26 (Fast)": "tracking-runners/7",
    # Model Size Test (Medium & Large)
    "RF-DETR (Medium)": "tracking-runners/4",
    "RF-DETR (Large)": "tracking-runners/6"
}

# Hyperparameter Grid Search
CONFIDENCE_THRESHOLDS = [0.35, 0.45, 0.55]
LOST_TRACK_BUFFER = 30 # Statisch gehouden op basis van eerdere optimalisatie

# Video-eigenschappen ophalen
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
total_frames = video_info.total_frames

# Evaluatie Loop
for model_name, model_id in MODELS_TO_TEST.items():
    print(f"\n{'='*60}\nEVALUATIE ARCHITECTUUR: {model_name} (ID: {model_id})\n{'='*60}")
    try:
        model = get_model(model_id=model_id, api_key=ROBOFLOW_API_KEY)
    except Exception as e:
        print(f"Fout bij laden model {model_name}: {e}")
        continue

    for ct in CONFIDENCE_THRESHOLDS:
        tracker = sv.ByteTrack(lost_track_buffer=LOST_TRACK_BUFFER)
        unique_ids = set()
        start_time = time.time()
        cap = cv2.VideoCapture(VIDEO_PATH)

        for frame_idx in tqdm(range(total_frames), desc=f"Testen CT={ct:.2f}"):
            success, frame = cap.read()
            if not success: break

            results = model.infer(frame)[0]
            detections = sv.Detections.from_inference(results)
            detections = detections[detections.confidence > ct]
            tracks = tracker.update_with_detections(detections=detections)

            if tracks.tracker_id is not None:
                unique_ids.update(tracks.tracker_id)

        cap.release()
        end_time = time.time()

        processing_time = end_time - start_time
        fps = total_frames / processing_time
        counted_runners = len(unique_ids)
        error_margin = abs(counted_runners - GROUND_TRUTH_COUNT)

        print(f"  > CT: {ct:.2f} | Geteld: {counted_runners:>4} | "
              f"Foutmarge: {error_margin:>3} | Snelheid: {fps:>5.2f} FPS | "
              f"Verwerkingstijd: {processing_time:>6.1f} sec")

## Hyperparameter Test - YOLOv11 (Fast)
Test van de Lost Track Buffer variabele op de optimale Confidence Threshold.

In [ ]:
print("Start hyperparameter test voor YOLOv11 (Fast)...\n")
MODEL_NAME = "YOLOv11 (Fast)"
MODEL_ID = "tracking-runners/8"
CT = 0.55
LTB_VALUES = [15, 30, 60]

video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
total_frames = video_info.total_frames
print(f"Model: {MODEL_NAME} (ID: {MODEL_ID}) | CT = {CT}\n")

model = get_model(model_id=MODEL_ID, api_key=ROBOFLOW_API_KEY)

for ltb in LTB_VALUES:
    print(f"--- Lost Track Buffer = {ltb} ---")
    tracker = sv.ByteTrack(lost_track_buffer=ltb)
    unique_ids = set()
    start_time = time.time()
    cap = cv2.VideoCapture(VIDEO_PATH)
    
    for frame_idx in tqdm(range(total_frames), desc=f"LTB={ltb}"):
        success, frame = cap.read()
        if not success:
            break
            
        results = model.infer(frame)[0]
        detections = sv.Detections.from_inference(results)
        detections = detections[detections.confidence > CT]
        tracks = tracker.update_with_detections(detections=detections)
        
        if tracks.tracker_id is not None:
            unique_ids.update(tracks.tracker_id)
            
    cap.release()
    end_time = time.time()
    
    processing_time = end_time - start_time
    fps = total_frames / processing_time
    counted_runners = len(unique_ids)
    error_margin = abs(counted_runners - GROUND_TRUTH_COUNT)
    
    print(f"  Geteld: {counted_runners:>4} | Foutmarge: {error_margin:>3} | "
          f"Snelheid: {fps:>5.2f} FPS | Tijd: {processing_time:>6.1f} sec\n")

## Visuele Validatie

In [ ]:
print("\nStart met het extraheren van het validatie-frame...")
OPTIMAL_CT = 0.55
TARGET_FRAME = 400
BEST_MODEL_ID = "tracking-runners/8" # YOLOv11

model = get_model(model_id=BEST_MODEL_ID, api_key=ROBOFLOW_API_KEY)
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, TARGET_FRAME) 
success, frame = cap.read()
cap.release()

if not success:
    print(f"Fout: Frame {TARGET_FRAME} kon niet worden ingeladen.")
else:
    results = model.infer(frame)[0]
    detections = sv.Detections.from_inference(results)
    detections = detections[detections.confidence > OPTIMAL_CT]
    
    box_annotator = sv.BoxAnnotator(thickness=3)
    annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
    
    annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 9))
    plt.imshow(annotated_frame_rgb)
    plt.axis('off')
    plt.tight_layout()
    plt.show()